# Hole — Qwen2.5-Coder-1.5B smoke fine-tune (Colab)

Repo: [pendragonIV/Hole](https://github.com/pendragonIV/Hole). Bật **GPU**.

1. Chạy ô cài đặt → **Runtime → Restart runtime** → chạy tiếp từ Mount Drive.
2. Cài đặt theo notebook gốc [Qwen2.5 (7B) Alpaca](https://github.com/unslothai/notebooks/blob/main/nb/Qwen2.5_(7B)-Alpaca.ipynb); model: `Qwen/Qwen2.5-Coder-1.5B-Instruct` (xem `docs/QUYET_DINH_VA_THUC_HIEN.md`).


In [ ]:
!nvidia-smi


### Cài Unsloth

Sau ô dưới: **Restart runtime**, rồi chạy lại từ ô **Mount Drive**.


In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2


### Sau restart — Mount Drive + OUTPUT_DIR


In [ ]:
from google.colab import drive
import os

OUTPUT_DIR = "/content/drive/MyDrive/qwen-ft/hole-qwen25-coder-1.5b"
try:
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount:", e)
    OUTPUT_DIR = "/content/qwen_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("OUTPUT_DIR =", OUTPUT_DIR)


### HF token (Colab Secrets: HF_TOKEN)


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass


### Model + LoRA + dataset + SFTTrainer


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

from datasets import load_dataset
dataset = load_dataset("unsloth/alpaca-cleaned", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = OUTPUT_DIR,
        report_to = "none",
    ),
)


### Train (60 steps — smoke)


In [ ]:
trainer_stats = trainer.train()
print(trainer_stats)


### Lưu LoRA


In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/lora_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_model")
print("Saved to", f"{OUTPUT_DIR}/lora_model")
